# Lab 21 — Failure Analysis and Safety Evaluation
**Goal:** Create a multi-dimensional safety/capability test set and inspect aggregate metrics versus failure buckets.

Free-first: CPU or free Colab where possible; expensive paths are clearly marked optional.



In [ ]:
tests=[
 {"id":"cap1","bucket":"capability","pass":True,"severity":1},
 {"id":"safe1","bucket":"privacy","pass":False,"severity":5},
 {"id":"safe2","bucket":"harmful_request","pass":True,"severity":5},
 {"id":"robust1","bucket":"robustness","pass":False,"severity":3},
 {"id":"cap2","bucket":"capability","pass":True,"severity":1},
]
print(tests)


In [ ]:
from collections import defaultdict
stats=defaultdict(list)
for t in tests: stats[t["bucket"]].append(t["pass"])
for k,v in stats.items(): print(k,sum(v)/len(v))


In [ ]:
# A single score can hide severe failures.
overall=sum(t["pass"] for t in tests)/len(tests)
weighted=sum(int(t["pass"])*t["severity"] for t in tests)/sum(t["severity"] for t in tests)
print("overall:",overall,"severity-weighted:",weighted)


### Answer
Evaluation must reflect deployment risk. A high aggregate score can coexist with an unacceptable severe failure. Keep bucket-level metrics, severity, representative examples, and qualitative failure analysis.


## Primary source
https://arxiv.org/abs/2203.02155


## Deepening: failures become intervention maps

This section turns the notebook into a quantitative lab. Predict before executing.


In [ ]:
from collections import Counter

failures = [
    {"type":"capability","severity":"S1"},
    {"type":"grounding","severity":"S2"},
    {"type":"format","severity":"S0"},
    {"type":"grounding","severity":"S2"},
    {"type":"safety","severity":"S3"},
    {"type":"robustness","severity":"S1"},
]
print("by type:", Counter(x["type"] for x in failures))
print("by severity:", Counter(x["severity"] for x in failures))

# A small cross-tab for prioritization.
for t in sorted(set(x["type"] for x in failures)):
    n = sum(x["type"]==t for x in failures)
    print(t, "count=", n)


## Break experiment

Add one failure family that is rare but severe.

Ask why counting failures alone is insufficient for release decisions.

### Required report

failure → severity → confirming test → owning system layer → mitigation → regression test

Complete templates/EXPERIMENT_CARD.md for the final result.


## 7. Research exercise — safety slices on real instruction data


In [ ]:
from datasets import load_dataset
from collections import Counter

ds=load_dataset("CohereLabs/aya_dataset",split="train",streaming=True)
rows=[]
for row in ds:
    p=str(row.get("inputs","")); t=str(row.get("targets",""))
    lang=row.get("language") or row.get("language_code")
    if p and t: rows.append((lang,p,t))
    if len(rows)>=1000: break

# Simple, transparent slice probes; these are not safety labels.
terms=["password","violence","medical","politics","religion","code"]
for term in terms:
    hits=sum(term.lower() in (p+" "+t).lower() for _,p,t in rows)
    print(term, "keyword_hits=",hits)
print("languages:",Counter(l for l,_,_ in rows).most_common(8))


<details><summary><strong>Expected findings / answer</strong></summary>

Keyword counts are diagnostic probes, not safety judgments. They reveal how much potentially sensitive or domain-specific material appears in a sample and where language slices may differ, but they do not tell you whether an example is harmful, benign, or correctly handled. A robust safety evaluation needs explicit labels, attack sets, policy criteria, and human/model review. The research lesson is **do not mistake a cheap proxy for a safety evaluation**.
</details>
